In [46]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from scipy.stats import linregress
from pathlib import Path
from abc import ABCMeta, abstractmethod
from time import time

In [47]:
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.util import Logger, Util

In [48]:
# import importlib
# import configs.config
# importlib.reload(configs.config)
# from configs.config import *

In [49]:
pd.set_option("display.max_columns",200)
pd.set_option("display.max_rows", 500)

# 基底クラス

In [50]:
def decorate(s: str, decoration=None):
    if decoration is None:
        decoration = '★' * 20

    return ' '.join([decoration, str(s), decoration])

class Timer:
    def __init__(self, logger=None, format_str='{:.3f}[s]', prefix=None, suffix=None, sep=' ', verbose=0):

        if prefix: format_str = str(prefix) + sep + format_str
        if suffix: format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None
        self.verbose = verbose

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        if self.verbose is None:
            return
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

In [51]:
class FeatureBase(metaclass=ABCMeta):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        self.use_cache = use_cache
        self.name = self.__class__.__name__
        self.cache_dir = Path(DIR_FEATURE)
        self.logger = logger
        self.seve_cache = save_cache
        self.use_cols = None
        self.key_column = None
    
    # 共通のキー整形 & 重複チェック
    def enforce_key_integrity(self, df: pd.DataFrame) -> pd.DataFrame:
        for key in self.key_column:
            if key not in df.columns:
                raise KeyError(f"{self.name}: キーカラム '{key}' が存在しません")
        assert ~df[self.key_column].duplicated().any(), f"{self.name}: 主キー {self.key_column} に重複があります"
    
    @abstractmethod
    def _create_feature(self) -> pd.DataFrame:
        """
        特徴量生成の実装をサブクラスで定義する必要があります。
        :return: pd.DataFrame 生成された特徴量
        """
        raise NotImplementedError()

    # 特徴量生成処理
    def create_feature(self) -> pd.DataFrame:

        # クラス名.pkl
        file_name = os.path.join(self.cache_dir, f"{self.name}.pkl")

        # キャッシュを使う & ファイルがあるなら読み出し
        if os.path.isfile(str(file_name)) and self.use_cache:
            feature = pd.read_pickle(file_name)

        # 変換処理を実行
        else:
            # train/testの区別なく変換処理を実行
            feature = self._create_feature()

            # 主キーチェック
            if self.key_column is not None:
                self.enforce_key_integrity(feature)

            # 保存する場合
            if self.seve_cache:
                feature.to_pickle(file_name)

        return feature

In [52]:
def one_hot_encode(df, col, drop_col=True):
    """
    特定の列に対してOne-Hotエンコーディングを適用します。
    
    :param df: pd.DataFrame 対象のDataFrame
    :param col: str エンコードする列名
    :return: pd.DataFrame エンコードされたDataFrame
    """
    # Initialize OneHotEncoder
    encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

    # Fit and transform the specified column
    encoded = encoder.fit_transform(df[[col]])

    # Convert the encoded array to a DataFrame
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out([col]))

    # concat
    df_concat = pd.concat([df, encoded_df], axis=1)

    # drop 
    if drop_col:
        df_concat.drop(columns=col, inplace=True)

    return df_concat

In [53]:
def clean_feature_names(data):
    # 特徴量名を修正
    data.columns = data.columns.str.replace(r'[^\w]', '_', regex=True)
    return data

# 特殊文字をアンダースコアに置換
def replace_special_characters(text):
    """
    特徴量名から特殊文字を削除し、LightGBMがサポートする形式に変換する。
    """
    text = text.replace(r'[^\w]', '_', regex=True)  # 特殊文字をアンダースコアに置換
    return text

# 継承クラス

In [54]:
class Key(FeatureBase):
    """
    TrainFeatureクラスは、train.csvデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号', 'category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        train.csvデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # train.csvデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))
        df_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
        df_Key = pd.concat([df_train, df_test], ignore_index=True)[self.key_column]

        return df_Key

In [55]:
class Target(FeatureBase):
    """
    Targetクラスは、ターゲットデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号', 'category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        ターゲットデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成されたターゲットデータを含むDataFrame。
        """
        # ターゲットデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))

        # 必要なカラムを選択
        df_target = df_train[['社員番号', 'category', 'target']]

        # 主キーとターゲット列を含むDataFrameを返す
        return df_target

In [56]:
class CategoryFeature(FeatureBase):
    """
    TrainFeatureクラスは、train.csvデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        train.csvデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # train.csvデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))
        df_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
        df_all = pd.concat([df_train, df_test], ignore_index=True)

        df_category_feature = df_all.copy().drop_duplicates('category')[self.key_column]

        # One-hotエンコーディング
        # df_category_feature = one_hot_encode(df_category_feature, 'category', False)

        # labaelエンコーディング
        le = LabelEncoder()
        df_category_feature['le_category'] = le.fit_transform(df_category_feature['category'])

        # 主キーとターゲット列を含むDataFrameを返す
        return df_category_feature

In [57]:
class CareerFeature(FeatureBase):
    """
    CareerBlockクラスは、キャリアデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        キャリアデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 前処理済みのキャリアデータを読み込む
        df_career = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_career.pkl"))

        df_career_feature = df_career.copy()

        # ポジティブな回答数
        # df_career_feature['positive_responses'] = df_career_feature.iloc[:, 1:].apply(lambda row: (row == 1).sum(), axis=1)

        # # ネガティブな回答数
        # df_career_feature['negative_responses'] = df_career_feature.iloc[:, 1:].apply(lambda row: (row == 0).sum(), axis=1)

        # # ポジティブな回答の割合
        # df_career_feature['positive_ratio'] = df_career_feature['positive_responses'] / (df_career_feature.shape[1] - 1)

        return df_career_feature

In [58]:
   
class UdemyActivityFeature(FeatureBase):
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:

        # 前処理済みのUdemy活動データを読み込む
        df_udemy = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_udemy_activity.pkl"))

        df_udemy_feature = df_udemy.copy()[self.key_column].drop_duplicates()

        # クイズ判定
        df_udemy["is_quiz"] = df_udemy["レクチャーもしくはクイズ"]=="Quiz"

        # 基本統計量の集計
        df_udemy_activity_numerical = df_udemy.groupby(self.key_column).agg(
            # 推定完了率%
            mean_推定完了率=('推定完了率_', 'mean'),
            min_推定完了率=('推定完了率_', 'min'),
            max_推定完了率=('推定完了率_', 'max'),
            std_推定完了率=('推定完了率_', 'std'),
            count_推定完了率=('推定完了率_', 'count'),
            # 最終結果（クイズの場合）
            mean_最終結果=('最終結果_クイズの場合_', 'mean'),
            min_最終結果=('最終結果_クイズの場合_', 'min'),
            max_最終結果=('最終結果_クイズの場合_', 'max'),
            std_最終結果=('最終結果_クイズの場合_', 'std'),
            count_最終結果=('最終結果_クイズの場合_', 'count'),
            # マーク済み修了
            mean_マーク済み修了=('マーク済み修了', 'mean'),
            min_マーク済み修了=('マーク済み修了', 'min'),
            max_マーク済み修了=('マーク済み修了', 'max'),
            std_マーク済み修了=('マーク済み修了', 'std'),
            count_マーク済み修了=('マーク済み修了', 'count'),
        )

        # コースカテゴリごとの回数を集計
        df_udemy_activity_course_category = df_udemy.pivot_table(
            index='社員番号',
            columns='コースカテゴリー',
            values='コースID',
            aggfunc='count',
            fill_value=0
        ).reset_index()
        # カラム名を変更
        prefix = "ua_コースカテゴリ_"
        df_udemy_activity_course_category.columns = [col if col=='社員番号' else prefix + col for col in df_udemy_activity_course_category.columns]

        # レクチャーもしくはクイズごとの回数を集計
        df_udemy_activity_type = df_udemy.pivot_table(
            index='社員番号',
            columns='レクチャーもしくはクイズ',
            values='コースID',
            aggfunc='count',
            fill_value=0
        ).reset_index()
        # カラム名を変更
        prefix = "ua_レクチャーもしくはクイズ_"
        df_udemy_activity_type.columns = [col if col=='社員番号' else prefix + col for col in df_udemy_activity_type.columns]

        # # コースIDごとの回数を集計
        # df_udemy_activity_course_id = df_udemy.pivot_table(
        #     index='社員番号',
        #     columns='コースID',
        #     values='コースID',
        #     aggfunc='count',
        #     fill_value=0
        # ).reset_index()     
        # # カラム名を変更
        # prefix = "ua_コースID_"
        # df_udemy_activity_course_id.columns = [str(col[0]) if col[0]=='社員番号' else prefix + str(col[1]) for col in df_udemy_activity_course_id.columns]



        # df_udemy_feature = df_udemy.groupby(self.key_column).agg(
        #     count_コースID=('コースID', 'count'),
        #     nunique_コースID=('コースID', 'nunique'),
        #     nunique_コースタイトル=('コースタイトル', 'nunique'),
        #     nunique_コースカテゴリ=('コースカテゴリー', 'nunique'),
        #     nunique_学習日数=('開始日', pd.Series.nunique),
        #     sum_マーク済み修了=('マーク済み修了', 'sum'),
        #     mean_推定完了率=('推定完了率%', 'mean'),
        #     min_開始日=('開始日', 'min'),
        #     max_開始日=('開始日', 'max'),
        #     rate_Quiz=('is_quiz', 'mean'),
        # ).reset_index()

        # # 学習スパン（日数）
        # df_udemy_feature["learning_span"] = (df_udemy_feature["max_開始日"] - df_udemy_feature["min_開始日"]).dt.days
        # # 日付型を数値型に変換
        # df_udemy_feature["min_開始日"] = df_udemy_feature["min_開始日"].apply(lambda x: float(datetime.strftime(x, format='%Y%m%d')))
        # df_udemy_feature["max_開始日"] = df_udemy_feature["max_開始日"].apply(lambda x: float(datetime.strftime(x, format='%Y%m%d')))

        # # クイズスコアの集計
        # df_quiz = df_udemy[df_udemy["is_quiz"]].copy()
        # df_quiz_stats = df_quiz.groupby(self.key_column).agg(
        #     count_クイズ=('最終結果（クイズの場合）', 'count'),
        #     mean_クイズスコア=('最終結果（クイズの場合）', 'mean'),
        # ).reset_index()

        # マージ
        df_udemy_feature = df_udemy_feature.merge(df_udemy_activity_numerical, on=self.key_column, how='left')
        df_udemy_feature = df_udemy_feature.merge(df_udemy_activity_course_category, on=self.key_column, how='left')
        df_udemy_feature = df_udemy_feature.merge(df_udemy_activity_type, on=self.key_column, how='left')

        # カラム名の修正
        df_udemy_feature = clean_feature_names(df_udemy_feature)

        return df_udemy_feature

In [59]:
class DxFeature(FeatureBase):
    """
    DxFeatureクラスは、DX関連のデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        DX関連データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 前処理済みのDXデータを読み込む
        df_dx = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_dx.pkl"))

        df_dx_feature = df_dx.copy()[self.key_column].drop_duplicates()

        # 研修カテゴリごとの参加回数を集計
        df_category_count = df_dx.pivot_table(
            index='社員番号',
            columns="研修カテゴリ",
            values="研修実施日",
            aggfunc="count",
            fill_value=0
        ).reset_index()
        # カラム名の変更
        prefix = "dx_研修カテゴリ_"
        df_category_count.columns = [col if col=='社員番号' else prefix + col for col in df_category_count.columns]

        # 研修名ごとの参加回数を集計
        df_name_count = df_dx.pivot_table(
            index='社員番号',
            columns="研修名",
            values="研修実施日",
            aggfunc="count",
            fill_value=0
        ).reset_index()
        # カラム名の変更
        prefix = "dx_研修名_"
        df_name_count.columns = [col if col=='社員番号' else prefix + col for col in df_name_count.columns]

        # 各社員の研修参加回数
        # df_dx_feature = df_dx.groupby(self.key_column).agg(
        #     count=('研修名', 'count'),
        #     unique_training_count=('研修名', 'nunique'),
        # ).reset_index()

        # 各社員のユニークな研修カテゴリ数
        # df_dx_feature['unique_training_categories'] = dx_data.groupby(self.key_column)['研修カテゴリ'].transform('nunique')

        # マージ
        df_dx_feature = df_dx_feature.merge(df_category_count, on=self.key_column, how='left')
        df_dx_feature = df_dx_feature.merge(df_name_count, on=self.key_column, how='left')
        
        # カラム名の修正
        df_dx_feature = clean_feature_names(df_dx_feature)

        return df_dx_feature
    

In [60]:
class HrFeature(FeatureBase):
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger=None)
        self.key_column = ['社員番号']

    def _create_feature(self) -> pd.DataFrame:

        df_hr = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_hr.pkl'))

        df_hr_feature = df_hr.copy()[self.key_column].drop_duplicates()

        # 研修カテゴリごとの参加回数を集計
        df_category_count = df_hr.pivot_table(
            index='社員番号',
            columns="カテゴリ",
            values="実施開始日",
            aggfunc="count",
            fill_value=0
        ).reset_index() 
        # カラム名の変更
        prefix = "hr_研修カテゴリ_"
        df_category_count.columns = [col if col=='社員番号' else prefix + col for col in df_category_count.columns]

        # 研修名ごとの参加回数を集計
        df_name_count = df_hr.pivot_table(
            index='社員番号',
            columns="研修名",
            values="実施開始日",
            aggfunc="count",
            fill_value=0
        ).reset_index()
        # カラム名の変更
        prefix = "hr_研修名_"
        df_name_count.columns = [col if col=='社員番号' else prefix + col for col in df_name_count.columns]

        # # 実施期間を算出（日数）
        # df_hr['研修日数'] = (df_hr['実施終了日'] - df_hr['実施開始日']).dt.days + 1
        # df_hr['研修日数'] = df_hr['研修日数'].fillna(1).clip(lower=1)

        # # 特徴量作成
        # df_hr_feature = df_hr.groupby("社員番号").agg(
        #     n_hr_total=("研修名", "count"),
        #     n_hr_unique_program=("研修名", "nunique"),
        #     n_hr_unique_category=("カテゴリ", "nunique"),
        #     first_hr_date=("実施開始日", "min"),
        #     last_hr_date=("実施終了日", "max"),
        #     n_hr_days=("研修日数", "sum"),
        # ).reset_index()

        # # 活動期間（最終日 - 初日）
        # df_hr_feature["hr_active_days"] = (df_hr_feature["last_hr_date"] - df_hr_feature["first_hr_date"]).dt.days
        # df_hr_feature.drop(["first_hr_date", "last_hr_date"], axis=1, inplace=True)

        # マージ
        df_hr_feature = df_hr_feature.merge(df_category_count, on=self.key_column, how='left')
        df_hr_feature = df_hr_feature.merge(df_name_count, on=self.key_column, how='left')

        # カラム名の修正
        df_hr_feature = clean_feature_names(df_hr_feature)

        return df_hr_feature

In [61]:
class OvertimeWorkByMonthFeature(FeatureBase):
    """
    OvertimeWorkFeatureクラスは、月ごとの残業データを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        残業データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 残業データを読み込む
        df_overtime = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_overtime_work_by_month.pkl"))

        # 特徴量例: 各社員の月ごとの平均残業時間
        df_overtime_feature = df_overtime.groupby(self.key_column).agg(
            avg_overtime_hours=('hours', 'mean'),
            max_overtime_hours=('hours', 'max'),
            min_overtime_hours=('hours', 'min'),
            # total_overtime_hours=('hours', 'sum'),
            std_overtime_hours=('hours', 'std'),
            count_overtime_months=('hours', 'count'),
        ).reset_index()

        return df_overtime_feature

In [62]:
class PositionHistoryFeature(FeatureBase):
    """
    PositionHistoryFeatureクラスは、役職履歴データを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        役職履歴データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 役職履歴データを読み込む
        df_position_history = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_position_history.pkl"))

        df_position_history_feature = df_position_history.copy()[self.key_column].drop_duplicates()

        # 勤務区分ごとの回数を集計
        df_work_type_count = df_position_history.pivot_table(
            index='社員番号',
            columns="勤務区分",
            values="year",
            aggfunc="count",
            fill_value=0
        ).reset_index()
        # カラム名の変更
        prefix = "ph_勤務区分_"
        df_work_type_count.columns = [col if col=='社員番号' else prefix + col for col in df_work_type_count.columns]

        # 役職ごとの回数を集計
        df_position_count = df_position_history.pivot_table(
            index='社員番号',
            columns="役職",
            values="year",
            aggfunc="count",
            fill_value=0
        ).reset_index()
        # カラム名の変更
        prefix = "ph_役職_"
        df_position_count.columns = [col if col=='社員番号' else prefix + col for col in df_position_count.columns]

        # # 特徴量例: 各社員の役職変更回数
        # df_position_history_feature = df_position_history.groupby(self.key_column).agg(
        #     position_change_count=('役職', 'nunique'),
        #     first_position=('役職', 'first'),
        #     last_position=('役職', 'last'),
        # ).reset_index()

        # # one-hotエンコーディング(OneHotEncoder)
        # df_position_history_feature = one_hot_encode(df_position_history_feature, 'first_position')
        # df_position_history_feature = one_hot_encode(df_position_history_feature, 'last_position')

        # マージ
        df_position_history_feature = df_position_history_feature.merge(df_work_type_count, on=self.key_column, how='left')
        df_position_history_feature = df_position_history_feature.merge(df_position_count, on=self.key_column, how='left')

        # カラム名の修正
        df_position_history_feature = clean_feature_names(df_position_history_feature)

        return df_position_history_feature

# 処理実行

In [63]:
def run_blocks(feature_blocks):
    print('start run blocks...')
    with Timer(prefix='run test'):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature()

In [64]:
feature_blocks = [
    Key(use_cache=False, save_cache=True, logger=None),
	Target(use_cache=False, save_cache=True, logger=None),
    CategoryFeature(use_cache=False, save_cache=True, logger=None),
	CareerFeature(use_cache=False, save_cache=True, logger=None),
	UdemyActivityFeature(use_cache=False, save_cache=True, logger=None),
	DxFeature(use_cache=False, save_cache=True, logger=None),
	HrFeature(use_cache=False, save_cache=True, logger=None),
	OvertimeWorkByMonthFeature(use_cache=False, save_cache=True, logger=None),
	PositionHistoryFeature(use_cache=False, save_cache=True, logger=None),
]

In [65]:
run_blocks(feature_blocks)

start run blocks...
	- <__main__.Key object at 0x00000202EDFC88B0> 0.083[s]
	- <__main__.Target object at 0x00000202EDFC8640> 0.031[s]
	- <__main__.CategoryFeature object at 0x00000202EDFC8730> 0.033[s]
	- <__main__.CareerFeature object at 0x00000202EDFC80A0> 0.038[s]
	- <__main__.UdemyActivityFeature object at 0x00000202EDFC8070> 0.837[s]
	- <__main__.DxFeature object at 0x00000202EDFC8E80> 0.058[s]
	- <__main__.HrFeature object at 0x00000202EDFC8190> 0.046[s]
	- <__main__.OvertimeWorkByMonthFeature object at 0x00000202EDFC8370> 0.068[s]
	- <__main__.PositionHistoryFeature object at 0x00000202EDFC8F70> 0.062[s]
run test 1.256[s]
